# TripMind hotel ranker (LightGBM LambdaRank)

Learns how real travellers choose hotels from **Expedia's Personalize Hotel Search** data
(search results with clicks and bookings), then orders TripMind's "Recommended" stays.

**Run on Kaggle:** create a notebook, *Add data* → competition
[expedia-personalized-sort](https://www.kaggle.com/c/expedia-personalized-sort/data)
(join the competition to accept its rules), then *Run all*. CPU is enough; a GPU is not used.

**Hand back:** download `model.txt` and `model_card.json` from `/kaggle/working` and put both in
`models/hotel_ranker/artifacts/`. TripMind only uses the model if the card's features match
and the model beats the rule-based baseline below.

Features are exactly those in `backend/app/modules/ml/features.py` (`HOTEL_RANKER_FEATURES`),
limited to what TripMind's own catalog can provide at serving time.

In [ ]:
import glob
import numpy as np, pandas as pd, lightgbm
path = glob.glob("/kaggle/input/**/train.csv", recursive=True)[0]
cols = ["srch_id", "prop_starrating", "prop_review_score", "price_usd", "srch_length_of_stay",
        "srch_adults_count", "srch_room_count", "srch_booking_window", "click_bool",
        "booking_bool", "random_bool"]
df = pd.read_csv(path, usecols=cols)
print(path, df.shape)

Only randomly ordered result lists (`random_bool == 1`) are used, so the model learns what people
choose, not the position Expedia's own ranker gave a hotel. Prices are clipped to remove obvious
errors; Expedia notes `price_usd` can be per night or per stay depending on the market, so it is
also divided by the stay length when it is far above the search's median.

In [ ]:
df = df[df.random_bool == 1].copy()
searches = df.srch_id.drop_duplicates().sample(n=min(150_000, df.srch_id.nunique()), random_state=7)
df = df[df.srch_id.isin(searches)]
df["price_usd"] = df.price_usd.clip(5, 5_000)
median = df.groupby("srch_id").price_usd.transform("median")
per_stay = (df.price_usd > 3 * median) & (df.srch_length_of_stay > 1)
df.loc[per_stay, "price_usd"] = df.price_usd / df.srch_length_of_stay
df["label"] = np.where(df.booking_bool == 1, 5, np.where(df.click_bool == 1, 1, 0))
df = df.sort_values("srch_id")
print(df.shape, df.label.value_counts().to_dict())

In [ ]:
FEATURES = ("star_rating", "review_score", "log_price_usd", "price_rank_pct", "star_rank_pct",
            "length_of_stay", "adults", "rooms", "booking_window_days")
g = df.groupby("srch_id")
X = pd.DataFrame({
    "star_rating": df.prop_starrating.astype(float),
    "review_score": df.prop_review_score.fillna(0).astype(float),
    "log_price_usd": np.log1p(df.price_usd),
    "price_rank_pct": g.price_usd.rank(pct=True, method="min").sub(g.price_usd.transform(lambda s: 1 / len(s))).clip(0, 1),
    "star_rank_pct": g.prop_starrating.rank(pct=True, method="min").sub(g.prop_starrating.transform(lambda s: 1 / len(s))).clip(0, 1),
    "length_of_stay": df.srch_length_of_stay.astype(float),
    "adults": df.srch_adults_count.astype(float),
    "rooms": df.srch_room_count.astype(float),
    "booking_window_days": df.srch_booking_window.astype(float),
})[list(FEATURES)]
ids = df.srch_id.unique()
rng = np.random.default_rng(7); rng.shuffle(ids)
valid_ids = set(ids[: len(ids) // 5])
is_valid = df.srch_id.isin(valid_ids).to_numpy()
def groups(mask): return df[mask].groupby("srch_id", sort=False).size().to_numpy()
train = lightgbm.Dataset(X[~is_valid], df.label[~is_valid], group=groups(~is_valid))
valid = lightgbm.Dataset(X[is_valid], df.label[is_valid], group=groups(is_valid))
params = {"objective": "lambdarank", "metric": "ndcg", "ndcg_eval_at": [5], "learning_rate": 0.05,
          "num_leaves": 63, "min_data_in_leaf": 200, "feature_fraction": 0.9, "verbose": -1}
model = lightgbm.train(params, train, 600, valid_sets=[valid],
                       callbacks=[lightgbm.early_stopping(50), lightgbm.log_evaluation(50)])

**Baseline:** TripMind's rule-based score (rating 50%, value 30%, stars 20%). The model must beat it.

In [ ]:
from sklearn.metrics import ndcg_score
v = df[is_valid].assign(model=model.predict(X[is_valid]),
                        rules=0.5 * X[is_valid].review_score / 5 + 0.3 * (1 - X[is_valid].price_rank_pct)
                              + 0.2 * X[is_valid].star_rating / 5)
def mean_ndcg(column):
    scores = [ndcg_score([grp.label], [grp[column]], k=5) for _, grp in v.groupby("srch_id") if grp.label.max() > 0]
    return float(np.mean(scores))
metrics = {"ndcg5_model": mean_ndcg("model"), "ndcg5_rules_baseline": mean_ndcg("rules"),
           "train_searches": float(len(ids) - len(valid_ids)), "valid_searches": float(len(valid_ids))}
print(metrics)
assert metrics["ndcg5_model"] > metrics["ndcg5_rules_baseline"], "model does not beat the baseline: do not ship"

In [ ]:
import json, datetime, lightgbm
def write_card(name, features, dataset, metrics, notes):
    version = datetime.date.today().strftime("%Y.%m.%d")
    card = {"name": name, "version": version,
            "trained_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
            "framework": "lightgbm", "framework_version": lightgbm.__version__,
            "dataset": dataset, "features": list(features), "metrics": metrics, "notes": notes}
    with open("/kaggle/working/model_card.json", "w") as f:
        json.dump(card, f, indent=2)
    print(json.dumps(card, indent=2))

    model.save_model("/kaggle/working/model.txt", num_iteration=model.best_iteration)
    write_card("hotel_ranker", FEATURES,
               "Expedia Personalize Hotel Search (Kaggle), randomly ordered result lists only",
               metrics, "LambdaRank; labels: booking 5, click 1. Prices may mix per-night and per-stay values.")